# Week 4 — Logical Reasoning for Planning

**CS F407 Artificial Intelligence · Lab 4**

A warehouse robot must deliver a package from location $A$ to location $C$. The robot can move only between connected locations ($A \leftrightarrow B \leftrightarrow C$), and it can pick up and drop the package. This notebook works through the lab's tasks:

- **Task 0:** specify the planning problem $(I, A, G)$.
- **Task 1:** build a plan by hand.
- **Task 2:** have an LLM implement a BFS planner.
- **Task 3:** test it.
- **Task 4:** separate the logic from the search.
- **Task 5:** compare the LLM's self-explanation with independent verification.
- **Tasks 6–8 (optional):** use **SWI-Prolog** as an independent plan verifier.

The answers are inline and also collected in [`answers-to-questions/answers.md`](../answers-to-questions/answers.md). The prompts are in [`prompts.md`](prompts.md), and the Prolog knowledge base is in [`planner.pl`](planner.pl).

**Central idea: Logic + Search = Planning.**

In [1]:
import subprocess
from collections import deque
from dataclasses import dataclass
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

def show(state):
    return "{" + ", ".join(sorted(state)) + "}"

## Task 0 — Understand the Planning Problem

Propositions are written as strings such as `"At(Robot,A)"`. A state is the **set of propositions that are true**. Anything not in the set is false (the closed-world assumption).

**(a) Initial state:** $I = \{\text{At}(Robot, A),\ \text{At}(Package, A)\}$

**(b) Goal:** $G = \{\text{At}(Package, C)\}$. A state $S$ satisfies the goal when $G \subseteq S$. Where the robot ends up is irrelevant.

**(c) Actions available to the robot:**
- $\text{Move}(A,B)$, $\text{Move}(B,A)$, $\text{Move}(B,C)$, $\text{Move}(C,B)$: only between connected locations. There is **no** $\text{Move}(A,C)$ or $\text{Move}(C,A)$.
- $\text{PickUp}(Package, L)$ for $L \in \{A, B, C\}$
- $\text{Drop}(Package, L)$ for $L \in \{A, B, C\}$

**(d) Preconditions and effects:**

| Action | Preconditions (positive) | Add effects | Delete effects |
|---|---|---|---|
| $\text{Move}(X, Y)$, for connected $X, Y$ | $\text{At}(Robot, X)$ | $\text{At}(Robot, Y)$ | $\text{At}(Robot, X)$ |
| $\text{PickUp}(Package, L)$ | $\text{At}(Robot, L)$, $\text{At}(Package, L)$ | $\text{Holding}(Package)$ | $\text{At}(Package, L)$ |
| $\text{Drop}(Package, L)$ | $\text{At}(Robot, L)$, $\text{Holding}(Package)$ | $\text{At}(Package, L)$ | $\text{Holding}(Package)$ |

None of these actions needs a *negative* precondition, but the representation supports them, as the prompt asked. One could add $\neg\text{Holding}(Package)$ to PickUp, but it is implied here: the package cannot be both held and at $L$, because PickUp deletes $\text{At}(Package, L)$.

In [2]:
@dataclass(frozen=True)
class Action:
    name: str
    pre_pos: frozenset = frozenset()     # must be true
    pre_neg: frozenset = frozenset()     # must be false
    add: frozenset = frozenset()         # positive effects
    delete: frozenset = frozenset()      # negative effects

    def applicable(self, state):
        # S |= Preconditions(a): every positive precondition holds and no negative one does.
        return self.pre_pos <= state and not (self.pre_neg & state)

    def apply(self, state):
        # Apply(S, a) = (S - Del(a)) | Add(a): remove negative effects, then add positive ones.
        return frozenset((state - self.delete) | self.add)

    def why_not(self, state):
        missing = sorted(self.pre_pos - state)
        violated = sorted(self.pre_neg & state)
        return missing, violated


def F(*props):
    return frozenset(props)


LOCATIONS = ["A", "B", "C"]
CONNECTIONS = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]

def move(x, y):
    return Action(f"Move({x},{y})", pre_pos=F(f"At(Robot,{x})"), add=F(f"At(Robot,{y})"), delete=F(f"At(Robot,{x})"))

def pickup(l):
    return Action(f"PickUp(Package,{l})", pre_pos=F(f"At(Robot,{l})", f"At(Package,{l})"),
                  add=F("Holding(Package)"), delete=F(f"At(Package,{l})"))

def drop(l):
    return Action(f"Drop(Package,{l})", pre_pos=F(f"At(Robot,{l})", "Holding(Package)"),
                  add=F(f"At(Package,{l})"), delete=F("Holding(Package)"))

INITIAL = F("At(Robot,A)", "At(Package,A)")
GOAL = F("At(Package,C)")
ACTIONS = [move(x, y) for x, y in CONNECTIONS] + [pickup(l) for l in LOCATIONS] + [drop(l) for l in LOCATIONS]
BY_NAME = {a.name: a for a in ACTIONS}

print(f"I = {show(INITIAL)}\nG = {show(GOAL)}\n{len(ACTIONS)} ground actions:")
for a in ACTIONS:
    print(f"  {a.name:<20} pre={show(a.pre_pos):<38} add={show(a.add):<22} del={show(a.delete)}")

I = {At(Package,A), At(Robot,A)}
G = {At(Package,C)}
10 ground actions:
  Move(A,B)            pre={At(Robot,A)}                          add={At(Robot,B)}          del={At(Robot,A)}
  Move(B,A)            pre={At(Robot,B)}                          add={At(Robot,A)}          del={At(Robot,B)}
  Move(B,C)            pre={At(Robot,B)}                          add={At(Robot,C)}          del={At(Robot,B)}
  Move(C,B)            pre={At(Robot,C)}                          add={At(Robot,B)}          del={At(Robot,C)}
  PickUp(Package,A)    pre={At(Package,A), At(Robot,A)}           add={Holding(Package)}     del={At(Package,A)}
  PickUp(Package,B)    pre={At(Package,B), At(Robot,B)}           add={Holding(Package)}     del={At(Package,B)}
  PickUp(Package,C)    pre={At(Package,C), At(Robot,C)}           add={Holding(Package)}     del={At(Package,C)}
  Drop(Package,A)      pre={At(Robot,A), Holding(Package)}        add={At(Package,A)}        del={Holding(Package)}
  Drop(Package,B)      pre={A

### Which actions are initially applicable?

In [3]:
for a in ACTIONS:
    missing, violated = a.why_not(INITIAL)
    status = "APPLICABLE" if a.applicable(INITIAL) else f"not applicable: missing {missing}"
    print(f"{a.name:<20} {status}")

Move(A,B)            APPLICABLE
Move(B,A)            not applicable: missing ['At(Robot,B)']
Move(B,C)            not applicable: missing ['At(Robot,B)']
Move(C,B)            not applicable: missing ['At(Robot,C)']
PickUp(Package,A)    APPLICABLE
PickUp(Package,B)    not applicable: missing ['At(Package,B)', 'At(Robot,B)']
PickUp(Package,C)    not applicable: missing ['At(Package,C)', 'At(Robot,C)']
Drop(Package,A)      not applicable: missing ['Holding(Package)']
Drop(Package,B)      not applicable: missing ['At(Robot,B)', 'Holding(Package)']
Drop(Package,C)      not applicable: missing ['At(Robot,C)', 'Holding(Package)']


**Question: starting from $I$, is $\text{PickUp}(Package, A)$ applicable? What about $\text{Drop}(Package, C)$?**

- **$\text{PickUp}(Package, A)$ is applicable.** Its preconditions are $\text{At}(Robot, A)$ and $\text{At}(Package, A)$, and both are in $I$, so $I \models \text{Preconditions}(\text{PickUp}(Package, A))$.
- **$\text{Drop}(Package, C)$ is not applicable.** It requires $\text{At}(Robot, C)$ and $\text{Holding}(Package)$, and *neither* holds in $I$: the robot is at $A$ and is not holding anything. Under the closed-world reading of the state, both propositions are false.

Only two actions are applicable in $I$ at all: $\text{PickUp}(Package, A)$ and $\text{Move}(A, B)$.

### Think About It — are all of its preconditions satisfied in the current state?

Being in the action list only says that an action *exists*. Whether it can be *executed* depends on the state. Checking $S \models \text{Preconditions}(a)$ is a logical entailment test: here, whether every precondition proposition is in $S$ and no negative precondition is. It is the step that stops a planner from, say, dropping a package it is not holding. Of the 10 ground actions, 8 are ruled out in $I$ by exactly this test.

## Task 1 — Construct a Plan by Hand

The lab suggests reasoning about $\text{Move}(A,B)$, $\text{PickUp}(Package,B)$, $\text{Move}(B,C)$ and $\text{Drop}(Package,C)$. **That sequence is not valid as written.** After $\text{Move}(A,B)$ the robot is at $B$, but the package is still at $A$, so $\text{PickUp}(Package, B)$ fails its precondition $\text{At}(Package, B)$. The robot has to pick the package up *before* leaving $A$.

The hand-constructed plan is therefore
$$\text{PickUp}(Package, A),\ \text{Move}(A, B),\ \text{Move}(B, C),\ \text{Drop}(Package, C).$$

| State | Facts | Action applied next | Why it is applicable |
|---|---|---|---|
| $S_0 = I$ | At(Robot,A), At(Package,A) | PickUp(Package,A) | At(Robot,A) ✓, At(Package,A) ✓ |
| $S_1$ | At(Robot,A), Holding(Package) | Move(A,B) | At(Robot,A) ✓ |
| $S_2$ | At(Robot,B), Holding(Package) | Move(B,C) | At(Robot,B) ✓ |
| $S_3$ | At(Robot,C), Holding(Package) | Drop(Package,C) | At(Robot,C) ✓, Holding(Package) ✓ |
| $S_4$ | At(Robot,C), At(Package,C) | — | $S_4 \models G$ ✓ |

Both sequences are checked by executing them against the action definitions. This independent validator is what every later test uses.

In [4]:
def validate_plan(plan, initial=INITIAL, goal=GOAL, actions=BY_NAME, verbose=True):
    # Independently execute a plan step by step. Returns (valid, trace).
    state, trace = initial, [("S0", None, initial)]
    for i, name in enumerate(plan, 1):
        a = actions.get(name)
        if a is None:
            if verbose: print(f"  step {i}: {name} is NOT an action of this problem")
            return False, trace
        if not a.applicable(state):
            missing, violated = a.why_not(state)
            if verbose: print(f"  step {i}: {name} NOT applicable in {show(state)}; missing {missing} {violated or ''}")
            return False, trace
        state = a.apply(state)
        trace.append((f"S{i}", name, state))
    ok = goal <= state
    if verbose:
        for label, name, s in trace:
            print(f"  {label}: {show(s)}" + (f"   <- {name}" if name else ""))
        print(f"  goal {show(goal)} satisfied: {ok}")
    return ok, trace

hand_plan = ["PickUp(Package,A)", "Move(A,B)", "Move(B,C)", "Drop(Package,C)"]
lab_example = ["Move(A,B)", "PickUp(Package,B)", "Move(B,C)", "Drop(Package,C)"]

print("Hand-constructed plan:")
validate_plan(hand_plan)
print("\nThe lab's example sequence, taken literally:")
validate_plan(lab_example)

Hand-constructed plan:
  S0: {At(Package,A), At(Robot,A)}
  S1: {At(Robot,A), Holding(Package)}   <- PickUp(Package,A)
  S2: {At(Robot,B), Holding(Package)}   <- Move(A,B)
  S3: {At(Robot,C), Holding(Package)}   <- Move(B,C)
  S4: {At(Package,C), At(Robot,C)}   <- Drop(Package,C)
  goal {At(Package,C)} satisfied: True

The lab's example sequence, taken literally:
  step 2: PickUp(Package,B) NOT applicable in {At(Package,A), At(Robot,B)}; missing ['At(Package,B)'] 


(False,
 [('S0', None, frozenset({'At(Package,A)', 'At(Robot,A)'})),
  ('S1', 'Move(A,B)', frozenset({'At(Package,A)', 'At(Robot,B)'}))])

## Task 2 — Ask an LLM to Implement the Planner

### Prompt used

The lab's suggested prompt was used verbatim, with the scenario (locations, initial state, goal and action descriptions) pasted in underneath it ([`prompts.md`](prompts.md), prompt 1):

> I want to implement a simple planning agent in Python. Represent a state as a set of logical propositions. Each action should contain: a name; positive preconditions; negative preconditions; positive effects; negative effects. An action is applicable if all of its preconditions are satisfied by the current state. When an action is applied: 1. remove its negative effects from the state; 2. add its positive effects to the state. Use breadth-first search to find a sequence of actions that achieves a specified goal. The program should also: detect when no plan exists; print the resulting sequence of actions; print the states reached after each action. Explain the implementation and identify any assumptions you make.

### The generated program (core, as returned)

The LLM's explanation listed its assumptions, including: *"the robot can move directly between any two locations; if your warehouse restricts movement, remove the corresponding Move actions."*

In [5]:
class LLMAction:
    def __init__(self, name, pos_pre, neg_pre, pos_eff, neg_eff):
        self.name = name
        self.pos_pre, self.neg_pre = set(pos_pre), set(neg_pre)
        self.pos_eff, self.neg_eff = set(pos_eff), set(neg_eff)

    def is_applicable(self, state):
        return self.pos_pre.issubset(state) and self.neg_pre.isdisjoint(state)

    def apply(self, state):
        new_state = set(state)
        new_state -= self.neg_eff        # 1. remove negative effects
        new_state |= self.pos_eff        # 2. add positive effects
        return frozenset(new_state)


def build_warehouse_actions(locations):
    actions = []
    for x in locations:
        for y in locations:
            if x != y:                   # assumption: robot can move between any two locations
                actions.append(LLMAction(f"Move({x},{y})", [f"At(Robot,{x})"], [], [f"At(Robot,{y})"], [f"At(Robot,{x})"]))
    for l in locations:
        actions.append(LLMAction(f"PickUp(Package,{l})", [f"At(Robot,{l})", f"At(Package,{l})"], [],
                                 ["Holding(Package)"], [f"At(Package,{l})"]))
        actions.append(LLMAction(f"Drop(Package,{l})", [f"At(Robot,{l})", "Holding(Package)"], [],
                                 [f"At(Package,{l})"], ["Holding(Package)"]))
    return actions


def bfs_plan(initial, goal, actions):
    start = frozenset(initial)
    frontier = deque([(start, [])])
    visited = {start}
    while frontier:
        state, plan = frontier.popleft()
        if goal.issubset(state):
            return plan
        for a in actions:
            if a.is_applicable(state):
                nxt = a.apply(state)
                if nxt not in visited:
                    visited.add(nxt)
                    frontier.append((nxt, plan + [a]))
    return None


def run_llm_planner(initial, goal, actions):
    plan = bfs_plan(initial, goal, actions)
    if plan is None:
        print("No plan found.")
        return None
    state = frozenset(initial)
    print("Plan found:")
    for i, a in enumerate(plan, 1):
        state = a.apply(state)
        print(f"  {i}. {a.name:<20} -> {show(state)}")
    return [a.name for a in plan]


llm_plan = run_llm_planner({"At(Robot,A)", "At(Package,A)"}, {"At(Package,C)"}, build_warehouse_actions(["A", "B", "C"]))

Plan found:
  1. PickUp(Package,A)    -> {At(Robot,A), Holding(Package)}
  2. Move(A,C)            -> {At(Robot,C), Holding(Package)}
  3. Drop(Package,C)      -> {At(Package,C), At(Robot,C)}


### Think About It — where does each part of the specification appear?

| Specification | In the generated program | Question it answers |
|---|---|---|
| **Preconditions** | `LLMAction.is_applicable`: `pos_pre ⊆ state` and `neg_pre ∩ state = ∅`; called in the `bfs_plan` loop before an action is used | When is an action applicable? |
| **Effects** | `LLMAction.apply`: `state − neg_eff`, then `∪ pos_eff`, in the order the prompt specified | How does the state change? |
| **Goal** | `goal.issubset(state)` at the top of the `bfs_plan` loop, checked when a state is dequeued | When does planning terminate? |
| **BFS** | `frontier = deque([...])` with `popleft()` (FIFO), a `visited` set of `frozenset` states, and plans stored alongside states | How are alternative plans explored? |

The *mechanics* all match the specification. The problem is in `build_warehouse_actions`. The comment `# assumption: robot can move between any two locations` generates $\text{Move}(A, C)$ and $\text{Move}(C, A)$, which the warehouse does not have. The planner therefore found a 3-step plan through a corridor that does not exist. The prompt never mentioned connectivity, so the LLM filled the gap with an assumption. It did disclose it, but in the explanation, not in the output.

## Task 3 — Test the Generated Planner

Each test records the initial state, the goal, whether a plan was found, the plan, and whether it is **actually valid**. Validity is checked by `validate_plan`, which executes the plan against the *specified* action set (from Task 0), not the planner's own.

In [6]:
def run_test(label, initial, goal, planner_actions, spec_actions, expect_plan):
    plan = bfs_plan(frozenset(initial), frozenset(goal), planner_actions)
    names = None if plan is None else [a.name for a in plan]
    valid = None if names is None else validate_plan(names, frozenset(initial), frozenset(goal), spec_actions, verbose=False)[0]
    passed = (names is None) if not expect_plan else bool(valid)
    print(f"--- {label}")
    print(f"  initial: {show(initial)}   goal: {show(goal)}")
    print(f"  plan found: {names is not None}   plan: {names}")
    print(f"  plan valid against the specification: {valid}")
    print(f"  => {'PASS' if passed else 'FAIL'}")
    return {"test": label, "initial": show(initial), "goal": show(goal), "found": names is not None,
            "plan": names, "valid": valid, "pass": passed}

def llm_actions(filter_fn=lambda a: True, extra=()):
    return [a for a in build_warehouse_actions(LOCATIONS) if filter_fn(a)] + list(extra)

def to_llm(a):
    return LLMAction(a.name, a.pre_pos, a.pre_neg, a.add, a.delete)

# Irrelevant actions for Test C: they move/affect only the robot.
BEEP = Action("Beep(Robot)", pre_pos=F("At(Robot,A)"))                   # no effect at all
PATROL = Action("Patrol(C)", pre_pos=F("At(Robot,C)"), add=F("Patrolled(C)"))

def spec(filter_fn=lambda a: True, extra=()):
    d = {a.name: a for a in ACTIONS if filter_fn(a)}
    d.update({a.name: a for a in extra})
    return d

no_pickup = lambda a: not a.name.startswith("PickUp")
print("===== LLM first attempt =====")
first_results = [
    run_test("Test A: solvable (original problem)", INITIAL, GOAL, llm_actions(), spec(), True),
    run_test("Test B: impossible (PickUp removed)", INITIAL, GOAL, llm_actions(no_pickup), spec(no_pickup), False),
    run_test("Test C: irrelevant actions; package cannot move, robot CAN reach C",
             INITIAL, GOAL, llm_actions(no_pickup, [to_llm(BEEP), to_llm(PATROL)]), spec(no_pickup, [BEEP, PATROL]), False),
    run_test("Test C2: irrelevant actions added to the solvable problem",
             INITIAL, GOAL, llm_actions(extra=[to_llm(BEEP), to_llm(PATROL)]), spec(extra=[BEEP, PATROL]), True),
]

===== LLM first attempt =====
--- Test A: solvable (original problem)
  initial: {At(Package,A), At(Robot,A)}   goal: {At(Package,C)}
  plan found: True   plan: ['PickUp(Package,A)', 'Move(A,C)', 'Drop(Package,C)']
  plan valid against the specification: False
  => FAIL
--- Test B: impossible (PickUp removed)
  initial: {At(Package,A), At(Robot,A)}   goal: {At(Package,C)}
  plan found: False   plan: None
  plan valid against the specification: None
  => PASS
--- Test C: irrelevant actions; package cannot move, robot CAN reach C
  initial: {At(Package,A), At(Robot,A)}   goal: {At(Package,C)}
  plan found: False   plan: None
  plan valid against the specification: None
  => PASS
--- Test C2: irrelevant actions added to the solvable problem
  initial: {At(Package,A), At(Robot,A)}   goal: {At(Package,C)}
  plan found: True   plan: ['PickUp(Package,A)', 'Move(A,C)', 'Drop(Package,C)']
  plan valid against the specification: False
  => FAIL


### What the tests found

- **Test A fails, and this is the important result.** The planner returned `PickUp(Package,A), Move(A,C), Drop(Package,C)`. It *looks* like a perfectly reasonable delivery plan, and it is even shorter than the hand plan. But $A$ and $C$ are not connected, so step 2 is not an action of this problem. The validator rejects it at step 2. The planner's own printout showed a valid-looking state after every step, because it was checking the plan against *its own* invented action set.
- **Test B passes:** with PickUp removed, the planner explores every reachable state and reports "No plan found" instead of inventing an action.
- **Test C passes:** the robot can reach $C$ (in both the planner's model and the real one), but the package cannot move, and the planner correctly does **not** treat $\text{At}(Robot, C)$ as achieving $\text{At}(Package, C)$. The goal test is a subset test on the goal propositions, not "is the robot at C?".
- **Test C2 fails for the same reason as Test A:** the irrelevant actions are correctly ignored, but the plan still uses $\text{Move}(A,C)$.

Test C is only convincing if it *could* have failed. So the planner was re-run with a deliberately wrong goal test, `At(Robot,C) in state` (a plausible mistake). Test C then catches it:

In [7]:
def bfs_plan_wrong_goal(initial, goal, actions):
    # Mutation: treats the robot reaching C as success.
    start = frozenset(initial)
    frontier, visited = deque([(start, [])]), {start}
    while frontier:
        state, plan = frontier.popleft()
        if "At(Robot,C)" in state:
            return plan
        for a in actions:
            if a.is_applicable(state) and (n := a.apply(state)) not in visited:
                visited.add(n); frontier.append((n, plan + [a]))
    return None

bad = bfs_plan_wrong_goal(INITIAL, GOAL, llm_actions(no_pickup, [to_llm(BEEP), to_llm(PATROL)]))
names = [a.name for a in bad]
print("mutated planner on Test C returns:", names)
print("valid against the specification?", validate_plan(names, INITIAL, GOAL, spec(no_pickup, [BEEP, PATROL]), verbose=False)[0],
      "-> Test C would FAIL, so the test is capable of catching this bug")

mutated planner on Test C returns: ['Move(A,C)']
valid against the specification? False -> Test C would FAIL, so the test is capable of catching this bug


### Fixing it

The fix belongs in the **specification**, not the search code. The prompt was updated to state the connectivity explicitly and to forbid adding assumed actions ([`prompts.md`](prompts.md), prompt 2):

> The warehouse has exactly these connections: A–B and B–C, in both directions. There is no direct connection between A and C. Generate Move actions **only** for these pairs. Do not add any action that is not listed in the problem. Also add a separate `validate_plan(plan)` function that re-executes a plan from the initial state, checks each action's preconditions against the state it is executed in, and reports the first failing step.

The regenerated action builder is the Task 0 `ACTIONS` list, grounded from `CONNECTIONS`. The search itself (`bfs_plan`) and the `apply` / `is_applicable` logic were accepted unchanged. Re-running all the tests:

In [8]:
SPEC_LLM_ACTIONS = [to_llm(a) for a in ACTIONS]
def fixed(filter_fn=lambda a: True, extra=()):
    return [a for a in SPEC_LLM_ACTIONS if filter_fn(a)] + [to_llm(e) for e in extra]

print("===== Corrected planner =====")
final_results = [
    run_test("Test A: solvable (original problem)", INITIAL, GOAL, fixed(), spec(), True),
    run_test("Test B: impossible (PickUp removed)", INITIAL, GOAL, fixed(no_pickup), spec(no_pickup), False),
    run_test("Test C: irrelevant actions; package cannot move, robot CAN reach C",
             INITIAL, GOAL, fixed(no_pickup, [BEEP, PATROL]), spec(no_pickup, [BEEP, PATROL]), False),
    run_test("Test C2: irrelevant actions added to the solvable problem",
             INITIAL, GOAL, fixed(extra=[BEEP, PATROL]), spec(extra=[BEEP, PATROL]), True),
    run_test("Test D: goal already true (package starts at C)", F("At(Robot,A)", "At(Package,C)"), GOAL, fixed(), spec(), True),
    run_test("Test E: robot and package start at C, goal At(Package,A)", F("At(Robot,C)", "At(Package,C)"),
             F("At(Package,A)"), fixed(), spec(), True),
]
assert all(r["pass"] for r in final_results)

===== Corrected planner =====
--- Test A: solvable (original problem)
  initial: {At(Package,A), At(Robot,A)}   goal: {At(Package,C)}
  plan found: True   plan: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
  plan valid against the specification: True
  => PASS
--- Test B: impossible (PickUp removed)
  initial: {At(Package,A), At(Robot,A)}   goal: {At(Package,C)}
  plan found: False   plan: None
  plan valid against the specification: None
  => PASS
--- Test C: irrelevant actions; package cannot move, robot CAN reach C
  initial: {At(Package,A), At(Robot,A)}   goal: {At(Package,C)}
  plan found: False   plan: None
  plan valid against the specification: None
  => PASS
--- Test C2: irrelevant actions added to the solvable problem
  initial: {At(Package,A), At(Robot,A)}   goal: {At(Package,C)}
  plan found: True   plan: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
  plan valid against the specification: True
  => PASS
--- Test D: goal already tr

In [9]:
print("Test A in full (the corrected planner's plan, independently executed):")
final_plan = final_results[0]["plan"]
ok, trace = validate_plan(final_plan)
with open(OUT_DIR / "test_results.md", "w") as f:
    f.write("| Test | Initial state | Goal | Plan found | Plan | Valid |\n|---|---|---|---|---|---|\n")
    for r in final_results:
        f.write(f"| {r['test']} | {r['initial']} | {r['goal']} | {r['found']} | {', '.join(r['plan']) if r['plan'] else ('[] (empty plan)' if r['plan'] == [] else 'No plan found')} | {r['valid']} |\n")
print("\nStates explored by BFS for Test A:", end=" ")
seen, q = {INITIAL}, deque([INITIAL])
while q:
    s = q.popleft()
    for a in ACTIONS:
        if a.applicable(s) and (n := a.apply(s)) not in seen:
            seen.add(n); q.append(n)
print(f"{len(seen)} reachable states (3 robot locations x 4 package situations: at A/B/C or held)")

Test A in full (the corrected planner's plan, independently executed):
  S0: {At(Package,A), At(Robot,A)}
  S1: {At(Robot,A), Holding(Package)}   <- PickUp(Package,A)
  S2: {At(Robot,B), Holding(Package)}   <- Move(A,B)
  S3: {At(Robot,C), Holding(Package)}   <- Move(B,C)
  S4: {At(Package,C), At(Robot,C)}   <- Drop(Package,C)
  goal {At(Package,C)} satisfied: True

States explored by BFS for Test A: 12 reachable states (3 robot locations x 4 package situations: at A/B/C or held)


**Test results (corrected planner):**

| Test | Initial state | Goal | Plan found? | Plan | Valid? |
|---|---|---|---|---|---|
| A. Solvable | At(Robot,A), At(Package,A) | At(Package,C) | yes | PickUp(Package,A), Move(A,B), Move(B,C), Drop(Package,C) | ✓ |
| B. Impossible (no PickUp) | At(Robot,A), At(Package,A) | At(Package,C) | **No plan found** | — | ✓ (correct) |
| C. Irrelevant actions, package stuck | At(Robot,A), At(Package,A) | At(Package,C) | **No plan found** (robot reaching C ≠ package at C) | — | ✓ (correct) |
| C2. Irrelevant actions, solvable | At(Robot,A), At(Package,A) | At(Package,C) | yes; Beep and Patrol not used | same 4 steps as A | ✓ |
| D. Goal already true | At(Robot,A), At(Package,C) | At(Package,C) | yes: the empty plan | [] | ✓ |
| E. Reverse delivery | At(Robot,C), At(Package,C) | At(Package,A) | yes | PickUp(Package,C), Move(C,B), Move(B,A), Drop(Package,A) | ✓ |

The first attempt passed B and C but **failed A and C2**: its plans were not valid in the real warehouse.

## Task 4 — Logic and Search

| Step | Component | Where in the code |
|---|---|---|
| Current state | — | the `state` popped from the BFS frontier |
| Check action preconditions | **logic**: does $S \models \text{Pre}(a)$? | `is_applicable` / `applicable` |
| **? → Select the applicable actions** (keep only the $a$ for which the check succeeds) | **logic** | the `if a.is_applicable(state)` filter in the loop |
| Generate successor state | **logic**: $S' = \text{Apply}(S, a) = (S \setminus \text{Del}(a)) \cup \text{Add}(a)$ | `apply` |
| Search over alternatives | **search**: which successor to explore next; FIFO order; skip visited states | `deque`, `popleft`, `visited` |
| Goal? | **logic**: does $S \models G$, i.e. $G \subseteq S$? If yes, stop and return the plan; if not, continue | `goal.issubset(state)` |

The missing step is **"select the applicable actions"**: the preconditions are checked, and only actions whose preconditions all hold go on to generate successors.

**How logic and search work together.** Logic answers local, one-state questions:
- *Can* this action be executed here?
- *What* is true after it?
- *Is* this state a goal?

Each is decided exactly from the action definitions. Search answers the global question: *which sequence* of these legal steps to try, in what order, while remembering which states have been seen. Logic turns the planning problem into a graph whose nodes are states and whose edges are applicable actions, and search finds a path through it. Neither is enough alone. Logic without search can only take one step. Search without logic would explore "moves" that break the rules, like dropping a package the robot does not hold. That is almost what happened with the LLM's invented $\text{Move}(A,C)$: an edge that the logic of the real warehouse does not contain.

### Think About It — logic determines what is possible; search determines what to try

The mapping to the previous module is direct:
- state sets are search states;
- applicable actions are successor functions;
- the goal subset test is the goal test;
- BFS is the same algorithm used for the warehouse grid, with the same completeness and shortest-plan guarantee for unit-cost actions.

What planning adds is a *logical representation* of states and actions. The successor function is derived from preconditions and effects instead of being hand-coded for a grid.

## Task 5 — Can the LLM Verify Its Own Plan?

The LLM was asked (prompt 3 in [`prompts.md`](prompts.md)):

> For every action in the plan, identify its preconditions and show that those preconditions are satisfied in the state in which the action is executed.

The question was asked for both of its plans. Its explanations, summarised:
- **First plan** (`PickUp(Package,A), Move(A,C), Drop(Package,C)`): the LLM declared it valid. "PickUp requires At(Robot,A) and At(Package,A), both true initially. Move(A,C) requires At(Robot,A), which is still true. Drop(Package,C) requires At(Robot,C) and Holding(Package), both true after step 2."
- **Corrected plan:** a correct step-by-step justification that agrees with the executed trace.

Now compare those explanations with the independently computed transitions:

In [10]:
print("Independent execution of the LLM's first plan against the warehouse specification:")
validate_plan(llm_plan)
print("\nIndependent execution of the corrected plan:")
_ = validate_plan(final_plan)

Independent execution of the LLM's first plan against the warehouse specification:
  step 2: Move(A,C) is NOT an action of this problem

Independent execution of the corrected plan:
  S0: {At(Package,A), At(Robot,A)}
  S1: {At(Robot,A), Holding(Package)}   <- PickUp(Package,A)
  S2: {At(Robot,B), Holding(Package)}   <- Move(A,B)
  S3: {At(Robot,C), Holding(Package)}   <- Move(B,C)
  S4: {At(Package,C), At(Robot,C)}   <- Drop(Package,C)
  goal {At(Package,C)} satisfied: True


**Which should you trust more, (a) the LLM's explanation or (b) the independently executed state transitions?**

**(b), the independently executed state transitions.** The LLM's explanation of its first plan was *internally consistent*: every precondition it cited really does hold, **given its own action model**. What it could not see was that its model was wrong. $\text{Move}(A,C)$ was never an action of this warehouse. An explanation generated by the same system, from the same assumptions, repeats the same error in fluent prose. It is a second opinion from the same source.

The independent executor is different in kind:
- it takes the actions from the *specification*;
- it computes every state mechanically, instead of describing them;
- it fails loudly, at the exact step and with the exact missing fact, when a precondition is not met.

It agreed with the LLM on the corrected plan and disagreed on the first one, and it was right both times. An explanation can help a human *understand* a plan, but only execution against the true model *verifies* it.

> **A generated explanation is not the same as an independent verification.**

## Optional Extension — Prolog as a Logical Verifier (Tasks 6–8)

The knowledge base is [`planner.pl`](planner.pl). The queries below run in **SWI-Prolog**, called from Python so the results are recorded in this notebook.

In [11]:
def prolog(query, file="planner.pl"):
    # Run one query in SWI-Prolog; return its printed answer.
    goal = f"( {query} -> writeln(true) ; writeln(false) )"
    out = subprocess.run(["swipl", "-q", "-g", goal, "-t", "halt", file], capture_output=True, text=True, timeout=30)
    return (out.stdout + out.stderr).strip()

def prolog_all(template, query, file="planner.pl"):
    goal = f"forall({query}, (print({template}), nl))"
    out = subprocess.run(["swipl", "-q", "-g", goal, "-t", "halt", file], capture_output=True, text=True, timeout=30)
    return (out.stdout + out.stderr).strip()

print(subprocess.run(["swipl", "--version"], capture_output=True, text=True).stdout.strip())
print()
print("Task 6")
for q in ["can_move(a,b)", "can_move(a,c)"]:
    print(f"?- {q}.   {prolog(q)}")
print("all can_move(X,Y) solutions:\n" + prolog_all("X-Y", "can_move(X,Y)"))

SWI-Prolog version 10.0.2 for arm64-darwin

Task 6


?- can_move(a,b).   true
?- can_move(a,c).   false
all can_move(X,Y) solutions:
a-b
b-a
b-c
c-b


### Task 6 — questions

**(a) Why does Prolog return `true` for `can_move(a,b)`?**
To prove `can_move(a,b)`, Prolog finds the rule `can_move(X,Y) :- connected(X,Y)`, unifies `X = a, Y = b`, and reduces the goal to `connected(a,b)`. That matches a **fact** in the knowledge base, so the proof succeeds: $\text{Connected}(a,b)$ together with $\text{Connected}(X,Y) \to \text{CanMove}(X,Y)$ entails $\text{CanMove}(a,b)$.

**(b) Why does it not establish `can_move(a,c)`?**
The only rule for `can_move` needs `connected(a,c)`, and there is no such fact or any rule that derives it. Prolog tries every clause, finds no proof, and answers `false`. Note what `false` means here: **"not provable from this knowledge base"**. This is negation as failure under the closed-world assumption, not a proof that $a$ and $c$ are disconnected in the real world. The rule also captures only *direct* moves. $c$ *is* reachable from $a$ in two steps, but `can_move` does not express that; the `reachable/2` extension below does.

**(c) What is the relationship between the rule `can_move(X,Y)` and $\text{Connected}(X,Y) \to \text{CanMove}(X,Y)$?**
The rule `can_move(X,Y) :- connected(X,Y).` is a **Horn clause**. It reads as the universally quantified implication
$$\forall X\, \forall Y\ \big(\text{Connected}(X,Y) \to \text{CanMove}(X,Y)\big),$$
with `:-` read as "if": the head `can_move(X,Y)` holds *if* the body `connected(X,Y)` holds. A query `?- can_move(a,b).` asks whether $\text{CanMove}(a,b)$ follows from the facts and rules. Prolog answers it by backward chaining: work back from the conclusion to facts through the implication. One difference from classical logic: Prolog treats the rule as the *only* way to derive `can_move` (the closed-world reading), so it behaves more like "CanMove *if and only if* Connected" for the purpose of failure.

In [12]:
print("Task 7")
for q in ["valid_move(a,b)", "valid_move(b,c)", "valid_move(a,c)"]:
    print(f"?- {q}.   {prolog(q)}")

print("\nChallenge: the Python planner proposes Move(a,c)")
print(f"?- valid_move(a,c).   {prolog('valid_move(a,c)')}   -> not supported by the warehouse knowledge")
print(f"?- reachable(a,c).    {prolog('reachable(a,c)')}   -> but c IS reachable in more than one move")

I_pl = "[at(robot,a), at(package,a)]"
G_pl = "[at(package,c)]"
checks = {
    "LLM first plan  [pickup(package,a), move(a,c), drop(package,c)]": "[pickup(package,a), move(a,c), drop(package,c)]",
    "corrected plan  [pickup(package,a), move(a,b), move(b,c), drop(package,c)]": "[pickup(package,a), move(a,b), move(b,c), drop(package,c)]",
    "lab example     [move(a,b), pickup(package,b), move(b,c), drop(package,c)]": "[move(a,b), pickup(package,b), move(b,c), drop(package,c)]",
}
print("\nFull-plan verification with valid_plan/3 (same STRIPS semantics, written independently in Prolog):")
for label, plan in checks.items():
    ok = prolog(f"valid_plan({I_pl}, {plan}, {G_pl})")
    why = "" if ok == "true" else "   first failing step: " + prolog_all("N-A", f"first_failure({I_pl}, {plan}, N, A)")
    print(f"  {label}: {ok}{why}")

print("\nPlans of length <= 4 that Prolog can find itself:")
print(prolog_all("P", f"(between(1,4,L), length(P,L), valid_plan({I_pl}, P, {G_pl}))"))

Task 7
?- valid_move(a,b).   true
?- valid_move(b,c).   true
?- valid_move(a,c).   false

Challenge: the Python planner proposes Move(a,c)
?- valid_move(a,c).   false   -> not supported by the warehouse knowledge
?- reachable(a,c).    true   -> but c IS reachable in more than one move

Full-plan verification with valid_plan/3 (same STRIPS semantics, written independently in Prolog):


  LLM first plan  [pickup(package,a), move(a,c), drop(package,c)]: false   first failing step: 2-move(a,c)
  corrected plan  [pickup(package,a), move(a,b), move(b,c), drop(package,c)]: true


  lab example     [move(a,b), pickup(package,b), move(b,c), drop(package,c)]: false   first failing step: 2-pickup(package,b)

Plans of length <= 4 that Prolog can find itself:
[pickup(package,a),move(a,b),move(b,c),drop(package,c)]


### Task 7 — results

| Query | Result | Reason |
|---|---|---|
| `valid_move(a,b)` | true | fact `connected(a,b)` |
| `valid_move(b,c)` | true | fact `connected(b,c)` |
| `valid_move(a,c)` | **false** | no `connected(a,c)` fact, and nothing derives it |

**Challenge: the Python planner proposes $\text{Move}(a,c)$.** Prolog answers `valid_move(a,c)` → `false`, so the proposed action is **not supported** by the warehouse knowledge. This is exactly the action that the LLM's first planner invented. The Prolog check rejects it for the right reason: it checks against the stated knowledge, not against the planner's own action list.

The `valid_plan/3` extension goes further and checks a whole plan with the STRIPS semantics:
- it rejects the LLM's first plan at step 2 (`move(a,c)`);
- it rejects the lab's literal example at step 2 (`pickup(package,b)`);
- it accepts the corrected plan.

Asked to find plans itself, it returns exactly one plan of length ≤ 4, the same one BFS found.

### Think About It — generate → independent verification

Python (with LLM help) *generated* the candidate plan. Prolog, with its own encoding of the warehouse facts and action rules, *checked* it. The two share no code, so a bug in one (the invented $\text{Move}(A,C)$) is not automatically repeated in the other. For the check to be meaningful, the verifier's knowledge must come from the specification, not from the generator.

In [13]:
print("Task 8")
print(f"?- reduce_speed.   {prolog('reduce_speed')}")

Task 8
?- reduce_speed.   true


### Task 8 — why `?- reduce_speed.` succeeds

Prolog works backwards from the query:
- `reduce_speed` is the head of the rule `reduce_speed :- slippery.`, so it needs `slippery`;
- `slippery` is the head of `slippery :- wet_road.`, so it needs `wet_road`;
- `wet_road` is a **fact**, so the chain closes and the query succeeds.

As a sequence of implications:
$$\underbrace{\text{WetRoad}}_{\text{fact}} \;\Rightarrow\; \underbrace{\text{Slippery}}_{\text{rule: } \text{WetRoad} \to \text{Slippery}} \;\Rightarrow\; \underbrace{\text{ReduceSpeed}}_{\text{rule: } \text{Slippery} \to \text{ReduceSpeed}} \quad\therefore\ \text{ReduceSpeed (conclusion)}$$

This is two applications of modus ponens: from $\text{WetRoad}$ and $\text{WetRoad} \to \text{Slippery}$, derive $\text{Slippery}$; from that and $\text{Slippery} \to \text{ReduceSpeed}$, derive $\text{ReduceSpeed}$.

### Think About It — Prolog vs classical logic

Prolog's execution has operational features that classical logic does not:
- **Depth-first backtracking search:** clause order matters, and a naive recursive `reachable` over the symmetric `connected` facts loops forever. This is why the extension carries a visited list.
- **Negation as failure:** `\+ member(...)`, and `false` meaning "not provable".
- **No occurs-check by default.**

So Prolog is a *procedure* for a fragment of logic, not logic itself. For facts, Horn rules and ground queries like these, its answers coincide with classical entailment.

### Prolog reflection

**1. What is the difference between a Prolog fact and a Prolog rule?**
A **fact** (`connected(a,b).`, `wet_road.`) states something unconditionally true: a clause with a head and no body. A **rule** (`can_move(X,Y) :- connected(X,Y).`) states that the head is true *if* the body is: a conditional, a universally quantified implication $\text{body} \to \text{head}$. Facts are the knowledge base's data; rules are its general knowledge, from which new facts are inferred.

**2. How does a Prolog query correspond to asking whether something follows from a knowledge base?**
`?- Q.` asks whether $KB \models Q$: is $Q$ a logical consequence of the facts and rules? Prolog tries to *prove* $Q$ by backward chaining with unification. `true` means a proof was found, with variable bindings if $Q$ has variables. `false` means no proof exists from this $KB$. That is "not entailed (under the closed-world assumption)", not "proven false".

**3. Why might it be useful to use a Prolog program to verify a plan generated by a Python program?**
- It is an **independent** implementation: different language, different code, and knowledge written directly from the specification.
- It is **declarative**: the facts and rules can be read and checked by a human almost line by line against the problem description, which is much easier than auditing imperative search code.
- A failed check gives a concrete, explainable reason, for example "no `connected(a,c)`", or "first failing step 2: `move(a,c)`".
- The **verifier is far simpler** than the planner. Checking a given plan needs no search, so it is easier to trust.

**4. What advantage does an independent verifier provide when the original plan was generated with the help of an LLM?**
LLM-generated code can embed **unstated assumptions**, like "all locations are connected", that make its outputs self-consistent but wrong. Any self-check built from the same assumptions, including the LLM's own explanation, inherits them. An independent verifier encodes the specification separately, so it catches exactly those errors: it rejected $\text{Move}(a,c)$ immediately. It also moves trust from *"the LLM says it's fine"* to *"a checkable proof from stated facts"*, and the burden of correctness from the generator to a small, auditable checker.

> **An AI system can generate a candidate solution, while a separate logical system checks it.**

## Reflection Questions

**1. Why is it useful to specify action preconditions and effects before asking an LLM to write the planner?**
The preconditions and effects *are* the problem. The search code is generic. Writing them down first:
- gives a precise description to hand to the LLM;
- gives something to check its code against;
- gives the ground truth for an independent validator.

It also exposes gaps: this lab's suggested prompt did not say which locations are connected. The LLM filled that gap with "all locations are connected" and produced an invalid plan. Having specified $\text{Move}$ only for $A{-}B$ and $B{-}C$ in Task 0 is what made that mistake detectable.

**2. Give an example of an error that could occur if the planner failed to check an action's preconditions.**
Without the precondition check, $\text{Drop}(Package, C)$ could be applied in the initial state. The robot would "drop" a package it is not holding, at a location it is not at, and the resulting state would contain $\text{At}(Package, C)$, the goal. The planner would return a one-step "plan", `[Drop(Package,C)]`, that is physically impossible. Similarly, $\text{PickUp}(Package, B)$ could be applied when the package is at $A$ (the lab's own example sequence), or $\text{Move}(B, C)$ applied while the robot is still at $A$, teleporting it.

**3. Why is a plan that "looks reasonable" not necessarily a valid plan?**
Validity is a precise property: every action's preconditions must hold *in the exact state produced by the previous actions*, and the final state must satisfy the goal. "Looks reasonable" is a judgement about whether the steps make sense in general. Two examples from this lab:
- `Move(A,B), PickUp(Package,B), Move(B,C), Drop(Package,C)` reads naturally but fails at step 2: the package is not at $B$.
- The LLM's `PickUp(Package,A), Move(A,C), Drop(Package,C)` reads even better, but uses a move that does not exist.

Only step-by-step execution against the action definitions shows the difference.

**4. What did the LLM contribute to the implementation?**
- the complete planner skeleton: an action class with the four precondition and effect sets, the applicability test, the effect application in the specified order, and BFS over `frozenset` states with a visited set;
- the printing of the plan and states, and no-plan detection;
- an explanation that listed its assumptions;
- a step-by-step justification of the corrected plan.

All the search and state-update logic was correct and was accepted unchanged.

**5. What did you have to verify independently?**
- that the *action set* matched the warehouse, i.e. that no invented $\text{Move}(A,C)$ was present;
- that every returned plan executes correctly against the specification (`validate_plan`, and Prolog's `valid_plan/3`);
- that "No plan found" is reported when no plan exists, rather than an action being invented (Test B);
- that the goal test checks $\text{At}(Package, C)$ and not the robot's location (Test C, plus the mutation check showing that the test *can* fail);
- the edge cases (goal already true; the reverse delivery);
- the LLM's explanation of its own plan (Task 5).

**6. In this laboratory, where is logical reasoning being used?**
- Deciding **applicability**: $S \models \text{Preconditions}(a)$, which checks that the positive preconditions are in $S$ and the negative ones are not.
- Computing **effects**: $\text{Apply}(S, a) = (S \setminus \text{Del}) \cup \text{Add}$. Under the closed-world representation, this updates exactly which propositions are true.
- The **goal test**: $S \models G$.
- **Validating** a plan: a chain of entailment checks.
- **In Prolog**: deriving `can_move`, `valid_move`, `reachable`, `valid_plan` and `reduce_speed` from facts and rules by backward chaining, i.e. modus ponens.

**7. How is planning related to the search algorithms studied in the previous module?**
Planning *is* search in a state space whose structure is defined logically:
- states = sets of propositions;
- successors = applicable actions;
- goal test = $G \subseteq S$;
- step cost = 1 per action.

BFS here is the same algorithm as in the grid-navigation labs, and it inherits the same properties: complete, and it finds a shortest plan when actions have unit cost. Informed search carries over too: A\* with a heuristic such as "number of unsatisfied goal propositions" would work unchanged. The difference is where the successor function comes from. In the grid it was hand-coded (four directions); here it is *derived* from action schemas. That is what makes a planner general across problems.